Creation of database

In [ ]:
import sqlite3
import os
from faker import Faker
import random

fake = Faker()

def generate_row():
    """Generate a row of data for the large database."""
    return (
        fake.uuid4(),
        fake.name(),
        fake.email(),
        fake.address(),
        fake.phone_number(),
        fake.job(),
        fake.company(),
        fake.date_of_birth().strftime('%Y-%m-%d'),
        fake.ssn(),
        fake.credit_card_number(),
        fake.credit_card_expire(),
        fake.credit_card_provider(),
        fake.country(),            # bank_country
        fake.currency_code(),
        round(random.uniform(100, 10000), 2),
        fake.date_time_this_decade().strftime('%Y-%m-%d %H:%M:%S')
    )

def create_large_database(db_name, target_size_bytes):
    conn = sqlite3.connect(db_name)
    cur = conn.cursor()

    # Create the full schema with primary key
    cur.execute('''
        CREATE TABLE IF NOT EXISTS customers (
            id TEXT PRIMARY KEY,
            name TEXT,
            email TEXT,
            address TEXT,
            phone_number TEXT,
            job TEXT,
            company TEXT,
            date_of_birth TEXT,
            ssn TEXT,
            credit_card_number TEXT,
            credit_card_expire TEXT,
            credit_card_provider TEXT,
            bank_country TEXT,
            currency_code TEXT,
            amount REAL,
            transaction_date TEXT
        )
    ''')
    conn.commit()

    count = 0
    while os.path.getsize(db_name) < target_size_bytes:
        row = generate_row()
        cur.execute('''
            INSERT INTO customers (
                id, name, email, address, phone_number, job, company, date_of_birth, ssn,
                credit_card_number, credit_card_expire, credit_card_provider, bank_country,
                currency_code, amount, transaction_date
            ) VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?)
        ''', row)
        count += 1
        if count % 1000 == 0:
            conn.commit()
            print(f"{db_name}: Inserted {count} rows; size: {os.path.getsize(db_name)} bytes")

    conn.commit()
    print(f"{db_name} created with {count} rows. Final size: {os.path.getsize(db_name)} bytes")
    conn.close()



# Target sizes
target_sizes = {
    'large_database.db': 1073741824,     # 1 GB
    
}

# Create large database with full schema
print(f"\nCreating large_database.db...")
create_large_database('large_database.db', target_sizes['large_database.db'])


Continuation of database creation

In [ ]:
import sqlite3
import os
import random
from tqdm import tqdm

def estimate_avg_row_size(conn: sqlite3.Connection, sample_limit: int = 1000) -> float:
    """Estimate average size in bytes of a full 'customers' row."""
    cur = conn.cursor()
    cur.execute("SELECT * FROM customers LIMIT ?", (sample_limit,))
    rows = cur.fetchall()
    if not rows:
        return 0.0
    total = sum(len(str(val).encode("utf-8")) for row in rows for val in row)
    return total / len(rows)

def create_id_only_samples(
    large_db: str,
    out_dir: str,
    sample_sizes_mb: list[int]
):
    """
    For each size, create `sample_{size}mb.db` containing only:
        CREATE TABLE customer_ids(id TEXT PRIMARY KEY);
    with a number of IDs approximating that many MB of full rows.
    """
    # 1) Prepare absolute output directory
    out_dir_abs = os.path.abspath(out_dir)
    os.makedirs(out_dir_abs, exist_ok=True)

    # 2) Open large DB and fetch all IDs
    conn = sqlite3.connect(large_db)
    cur = conn.cursor()
    cur.execute("SELECT COUNT(*) FROM customers")
    total_ids = cur.fetchone()[0]

    all_ids = []
    for (cid,) in tqdm(
        conn.execute("SELECT id FROM customers"),
        total=total_ids,
        desc="Fetching all IDs"
    ):
        all_ids.append(cid)

    # 3) Estimate size of a full row
    avg_size = estimate_avg_row_size(conn)
    if avg_size <= 0:
        conn.close()
        raise RuntimeError("Could not estimate average row size")

    # 4) For each target size, sample IDs and write a new DB
    for size in tqdm(sample_sizes_mb, desc="Creating sample DBs"):
        # compute how many rows
        target_bytes = size * 1024 * 1024
        n_rows = min(len(all_ids), int(target_bytes / avg_size))
        sampled = random.sample(all_ids, n_rows)

        # path for this sample
        sample_path = os.path.join(out_dir_abs, f"sample_{size}mb.db")
        parent_dir = os.path.dirname(sample_path)
        os.makedirs(parent_dir, exist_ok=True)

        # remove old file if exists
        if os.path.exists(sample_path):
            os.remove(sample_path)

        # connect to sample DB
        samp_conn = sqlite3.connect(sample_path)
        samp_cur = samp_conn.cursor()
        samp_cur.execute("CREATE TABLE customer_ids(id TEXT PRIMARY KEY)")

        # write IDs with an inner progress bar
        insert_q = "INSERT INTO customer_ids(id) VALUES (?)"
        for cid in tqdm(sampled, desc=f"Writing IDs for {size}MB", leave=False):
            samp_cur.execute(insert_q, (cid,))
        samp_conn.commit()
        samp_conn.close()

    conn.close()
    print(f"\n✅ ID-only sample DBs written to '{out_dir_abs}'")

if __name__ == "__main__":
    LARGE_DB = "large_database.db"
    OUTPUT_DIR = "samples"
    SIZES_MB = [50, 100, 150, 200, 250, 500, 750, 1000]

    create_id_only_samples(LARGE_DB, OUTPUT_DIR, SIZES_MB)


New code

In [ ]:
import requests
import json
import sqlite3
import time
import csv
import os
from tqdm import tqdm

# API setup
url = "http://localhost:8000/v1/chat/completions"
headers = {"Content-Type": "application/json"}

execution_summary = []
nl_sql_log = []

def ask_model(prompt):
    data = {
        "model": "defog/llama-3-sqlcoder-8b",
        "temperature": 0.0001,
        "messages": [{"role": "user", "content": prompt}]
    }
    response = requests.post(url, headers=headers, data=json.dumps(data))
    if response.status_code == 200:
        model_output = response.json()["choices"][0]["message"]["content"]
        print("\n🔍 Model Raw Output:\n", model_output)
        return model_output.strip()
    else:
        raise Exception(f"API request failed: {response.status_code}")

def natural_language_to_sql(user_command):
    prompt = f"""
### Task:
Convert the following natural language command into a correctly formatted SQLite SQL query.

### Notes:
- Use SQLite-compatible functions only.
- Avoid EXTRACT, AGE, TO_DATE, DATE_PART, etc.
- Use julianday for date math if needed.
- Return ONLY the SQL query (no markdown, no explanation).

Table: customers
Columns:
  id, name, email, address, phone_number, job, company, date_of_birth,
  ssn, credit_card_number, credit_card_expire, credit_card_provider,
  bank_country, currency_code, amount, transaction_date

### User Input:
{user_command}

### SQL Output:
"""
    sql = ask_model(prompt)
    nl_sql_log.append([user_command, sql])
    if sql.strip().upper().startswith("SELECT") and sql.strip().endswith(";"):
        return sql.strip()
    else:
        print("⚠️ Warning: Invalid SQL returned. Using fallback.")
        return "SELECT * FROM customers LIMIT 5;"

def execute_sql(db_path, sql_query):
    conn = sqlite3.connect(db_path)
    cur = conn.cursor()
    start = time.time()
    cur.execute(sql_query)
    result = cur.fetchall()
    end = time.time()
    conn.close()
    return result, end - start

def aggregate_results(results):
    try:
        return sum(row[0] for row in results if isinstance(row[0], (int, float)))
    except:
        return None

def compute_relative_error(gold, sample):
    try:
        return abs(sample - gold) / gold * 100 if gold != 0 else None
    except:
        return None

def compute_speed_overhead(gold_time, sample_time):
    try:
        return ((sample_time / gold_time) - 1) * 100 if gold_time != 0 else None
    except:
        return None

def export_summary_results():
    with open("experiment_results.csv", "w", newline="") as f:
        writer = csv.writer(f)
        writer.writerow(["Query", "Database", "Raw Sample Result", "Scaled Result", "Execution Time (s)", "Relative Error (%)", "Speed Overhead (%)"])
        writer.writerows(execution_summary)

def export_query_txt():
    with open("queries_log.txt", "w") as f:
        for nl, sql in nl_sql_log:
            f.write(f"Natural Language: {nl}\nSQL Query: {sql}\n\n")

def export_error_overhead():
    with open("errors_overhead.csv", "w", newline="") as f:
        writer = csv.writer(f)
        writer.writerow(["Query", "Database", "Relative Error (%)", "Speed Overhead (%)"])
        for row in execution_summary:
            writer.writerow([row[0], row[1], row[5], row[6]])

def run_experiment(sql_query, large_db_path, sample_db_paths_with_size):
    print("\n🔍 Executing on full database...")
    gold_results, gold_time = execute_sql(large_db_path, sql_query)
    gold_aggregated = aggregate_results(gold_results)

    print(f"✅ Full DB Result: {gold_aggregated} | Time: {gold_time:.4f}s")

    for db_path, size_bytes in tqdm(sample_db_paths_with_size.items(), desc="🔁 Sample DBs"):
        scaling_factor = size_bytes / os.path.getsize(large_db_path)
        inverse_scaling = 1 / scaling_factor

        try:
            sample_results, sample_time = execute_sql(db_path, sql_query)
            raw_result = aggregate_results(sample_results)

            scaled_result = raw_result * inverse_scaling if raw_result is not None else None
            rel_error = compute_relative_error(gold_aggregated, scaled_result)
            speed_overhead = compute_speed_overhead(gold_time, sample_time)

            print(f"\n📁 {db_path}")
            print(f"Raw: {raw_result}, Scaled: {scaled_result:.2f} | Time: {sample_time:.4f}s")
            print(f"Relative Error: {rel_error:.2f}%" if rel_error is not None else "N/A")
            print(f"Speed Overhead: {speed_overhead:.2f}%" if speed_overhead is not None else "N/A")

            execution_summary.append([
                sql_query,
                db_path,
                raw_result,
                scaled_result,
                f"{sample_time:.4f}",
                f"{rel_error:.2f}" if rel_error is not None else None,
                f"{speed_overhead:.2f}" if speed_overhead is not None else None
            ])

        except Exception as e:
            print(f"❌ Failed on {db_path}: {e}")
            execution_summary.append([sql_query, db_path, None, None, None, None, None])

def main():
    large_db_path = "large_database.db"
    sample_db_paths_with_size = {
        "50mb_sample.db": 52428800,
        "100mb_sample.db": 104857600,
        "150mb_sample.db": 157286400,
        "200mb_sample.db": 209715200,
        "250mb_sample.db": 262144000
    }

    while True:
        user_input = input("\n🧠 Enter your query (or type 'exit'): ")
        if user_input.lower() == "exit":
            break

        sql_query = natural_language_to_sql(user_input)
        print(f"\n📝 SQL Query:\n{sql_query}")
        run_experiment(sql_query, large_db_path, sample_db_paths_with_size)

    export_summary_results()
    export_query_txt()
    export_error_overhead()

    print("\n✅ Saved:")
    print(" - experiment_results.csv (full details)")
    print(" - queries_log.txt (natural input + SQL)")
    print(" - errors_overhead.csv (relative error + speed overhead only)")

if __name__ == "__main__":
    main()


sampled ids

In [ ]:
import sqlite3
import os
import random
from tqdm import tqdm

def estimate_avg_row_size(conn: sqlite3.Connection, sample_limit: int = 1000) -> float:
    """Estimate average bytes per full row by sampling up to `sample_limit` rows."""
    cur = conn.cursor()
    cur.execute("SELECT * FROM customers LIMIT ?", (sample_limit,))
    rows = cur.fetchall()
    if not rows:
        return 0.0
    total_bytes = sum(
        len(str(val).encode("utf-8"))
        for row in rows
        for val in row
    )
    return total_bytes / len(rows)

def create_sample_databases(
    large_db_path: str,
    output_dir: str,
    sample_sizes_mb: list[int]
):
    """
    For each size in sample_sizes_mb, build a .db file containing only
    a table `customer_ids(id)` sized (in count) to approximate that size
    of full rows—but we only store the IDs.
    """
    os.makedirs(output_dir, exist_ok=True)
    conn = sqlite3.connect(large_db_path)

    # 1) Fetch and cache all IDs with a progress bar
    cur = conn.cursor()
    cur.execute("SELECT COUNT(*) FROM customers")
    total_ids = cur.fetchone()[0]
    all_ids = []
    for (row_id,) in tqdm(conn.execute("SELECT id FROM customers"), total=total_ids, desc="Fetching all IDs"):
        all_ids.append(row_id)

    # 2) Compute average row size once
    avg_size = estimate_avg_row_size(conn)
    if avg_size <= 0:
        raise RuntimeError("Unable to estimate average row size")

    # 3) For each target size, sample & write IDs
    for size_mb in tqdm(sample_sizes_mb, desc="Creating sample DBs"):
        # number of rows to hit approx size_mb
        target_bytes = size_mb * 1024 * 1024
        n_rows = min(len(all_ids), int(target_bytes / avg_size))
        sampled = random.sample(all_ids, n_rows)

        sample_db = os.path.join(output_dir, f"sample_{size_mb}mb.db")
        if os.path.exists(sample_db):
            os.remove(sample_db)

        # Create SQLite file with only IDs table
        sample_conn = sqlite3.connect(sample_db)
        sample_cur = sample_conn.cursor()
        sample_cur.execute("CREATE TABLE customer_ids(id INTEGER PRIMARY KEY)")

        # 4) Write IDs into file with its own progress bar
        insert_query = "INSERT INTO customer_ids(id) VALUES (?)"
        for sid in tqdm(sampled, desc=f"Writing IDs for {size_mb}MB", leave=False):
            sample_cur.execute(insert_query, (sid,))
        sample_conn.commit()
        sample_conn.close()

    conn.close()
    print(f"✅ Sample DBs created in '{output_dir}'")

if __name__ == "__main__":
    LARGE_DB = "large_database.db"
    OUTPUT_DIR = "samples"
    SIZES_MB = [50, 100, 150, 200, 250, 500, 1000]

    create_sample_databases(LARGE_DB, OUTPUT_DIR, SIZES_MB)


In [ ]:
import csv

input_file = "errors_overhead.csv"
output_file = "errors_overhead_cleaned.csv"

with open(input_file, "r", newline="") as infile, open(output_file, "w", newline="") as outfile:
    reader = csv.reader(infile)
    writer = csv.writer(outfile)

    headers = next(reader)
    writer.writerow(headers)

    for row in reader:
        query, db_name, rel_error, speed_overhead = row

        # Clean speed overhead
        if speed_overhead and speed_overhead != "None":
            try:
                speed_overhead_value = abs(float(speed_overhead.replace("%", "")))
                speed_overhead = f"{speed_overhead_value:.2f}"
            except:
                pass

        writer.writerow([query, db_name, rel_error, speed_overhead])


Choosing from database

In [4]:
import requests
import json
import sqlite3
import time
import csv
import os
from tqdm import tqdm

# API setup
API_URL = "http://localhost:8000/v1/chat/completions"
HEADERS = {"Content-Type": "application/json"}

execution_summary = []
nl_sql_log = []

def ask_model(prompt: str) -> str:
    data = {
        "model": "defog/llama-3-sqlcoder-8b",
        "temperature": 0.0001,
        "messages": [{"role": "user", "content": prompt}]
    }
    resp = requests.post(API_URL, headers=HEADERS, data=json.dumps(data))
    resp.raise_for_status()
    return resp.json()["choices"][0]["message"]["content"].strip()

def natural_language_to_sql(user_command: str) -> str:
    prompt = f"""
### Task:
Convert the following natural language command into a correctly formatted SQLite SQL query.

### Notes:
- Use SQLite-compatible functions only.
- Avoid EXTRACT, AGE, TO_DATE, DATE_PART, etc.
- Use julianday for date math if needed.
- Return ONLY the SQL query (no markdown, no explanation).

Table: customers
Columns:
  id, name, email, address, phone_number, job, company, date_of_birth,
  ssn, credit_card_number, credit_card_expire, credit_card_provider,
  bank_country, currency_code, amount, transaction_date

### User Input:
{user_command}

### SQL Output:
"""
    sql = ask_model(prompt)
    nl_sql_log.append([user_command, sql])
    if sql.strip().upper().startswith("SELECT") and sql.strip().endswith(";"):
        return sql.strip()
    print("⚠️ Warning: Invalid SQL returned. Using fallback.")
    return "SELECT * FROM customers LIMIT 5;"

def execute_sql(conn: sqlite3.Connection, query: str):
    cur = conn.cursor()
    start = time.time()
    cur.execute(query)
    rows = cur.fetchall()
    return rows, time.time() - start

def aggregate_results(rows):
    try:
        return sum(r[0] for r in rows if isinstance(r[0], (int, float)))
    except:
        return None

def compute_relative_error(true_val, approx_val):
    if true_val not in (None, 0) and approx_val is not None:
        return abs(approx_val - true_val) / abs(true_val) * 100
    return None

def compute_time_saved(full_time, sample_time):
    if full_time not in (None, 0) and sample_time is not None:
        return (full_time - sample_time) / full_time * 100
    return None

def export_summary_results():
    with open("experiment_results.csv", "w", newline="") as f:
        w = csv.writer(f)
        w.writerow([
            "SQL Query", "Sample DB", "Raw Result", "Scaled Result",
            "Execution Time (s)", "Relative Error (%)", "Time Saved (%)"
        ])
        w.writerows(execution_summary)

def export_query_txt():
    with open("queries_log.txt", "w") as f:
        for nl, sql in nl_sql_log:
            f.write(f"Natural Language: {nl}\nSQL Query: {sql}\n\n")

def export_error_overhead():
    with open("errors_overhead.csv", "w", newline="") as f:
        w = csv.writer(f)
        w.writerow(["SQL Query", "Sample DB", "Relative Error (%)", "Time Saved (%)"])
        for row in execution_summary:
            w.writerow([row[0], row[1], row[5], row[6]])

def run_experiment(sql_query: str, large_db: str, sample_db_paths: dict[str,int]):
    conn = sqlite3.connect(large_db)
    cur = conn.cursor()

    # get total rows for scaling
    cur.execute("SELECT COUNT(*) FROM customers")
    total_rows = cur.fetchone()[0]

    print("\n🔍 Executing on full database...")
    full_rows, full_time = execute_sql(conn, sql_query)
    true_val = aggregate_results(full_rows)

    for sample_path in tqdm(sample_db_paths, desc="🔁 Sample DBs"):
        # load only IDs from the small DB
        samp_conn = sqlite3.connect(sample_path)
        ids = [r[0] for r in samp_conn.execute("SELECT id FROM customer_ids")]
        samp_conn.close()

        # safely quote and join
        id_list = ",".join(
            "'" + str(i).replace("'", "''") + "'"
            for i in ids
        )

        # create a temp view on the large DB
        cur.execute("DROP VIEW IF EXISTS sampled_customers")
        cur.execute(
            f"CREATE TEMP VIEW sampled_customers AS "
            f"SELECT * FROM customers WHERE id IN ({id_list})"
        )

        # swap table name in the query
        sampled_query = sql_query.replace("FROM customers", "FROM sampled_customers")
        sample_rows, sample_time = execute_sql(conn, sampled_query)
        raw_result = aggregate_results(sample_rows)

        # scale and compute error/time saved...
        scale = total_rows / len(ids)
        scaled = raw_result * scale if raw_result is not None else None
        rel_err = compute_relative_error(true_val, scaled)
        time_saved = compute_time_saved(full_time, sample_time)

        # collect and print...
        execution_summary.append([
            sql_query,
            sample_path,
            raw_result,
            f"{scaled:.2f}" if scaled is not None else None,
            f"{sample_time:.4f}",
            f"{rel_err:.2f}" if rel_err is not None else None,
            f"{time_saved:.2f}" if time_saved is not None else None
        ])

    conn.close()


def main():
    large_db = "large_database.db"
    # map small DB path → dummy size (unused now)
    sample_dbs = {
        "samples/sample_50mb.db": 50,
        "samples/sample_100mb.db": 100,
        "samples/sample_150mb.db": 150,
        "samples/sample_200mb.db": 200,
        "samples/sample_250mb.db": 250,
    }

    while True:
        user_input = input("\n🧠 Enter your query (or type 'exit'): ")
        if user_input.lower() == "exit":
            break

        sql = natural_language_to_sql(user_input)
        print(f"\n📝 SQL Query:\n{sql}")

        run_experiment(sql, large_db, sample_dbs)

    export_summary_results()
    export_query_txt()
    export_error_overhead()

    print("\n✅ Saved:")
    print(" - experiment_results.csv")
    print(" - queries_log.txt")
    print(" - errors_overhead.csv")

if __name__ == "__main__":
    main()



🧠 Enter your query (or type 'exit'):  What is the average age of all customers (in years)?



📝 SQL Query:
SELECT AVG(julianday('now') - julianday(c.date_of_birth))/31536000 AS average_age FROM customers c;

🔍 Executing on full database...


🔁 Sample DBs: 100%|██████████| 5/5 [03:16<00:00, 39.22s/it]



🧠 Enter your query (or type 'exit'):  What is the average amount spent by customers whose currency code is ‘EUR’?



📝 SQL Query:
SELECT AVG(c.amount) AS average_amount FROM customers c WHERE c.currency_code = 'EUR';

🔍 Executing on full database...


...


🧠 Enter your query (or type 'exit'):  How many customers are there in the database?



📝 SQL Query:
SELECT COUNT(*) FROM customers;

🔍 Executing on full database...


🔁 Sample DBs: 100%|██████████| 5/5 [11:18<00:00, 135.76s/it]



🧠 Enter your query (or type 'exit'):  exit



✅ Saved:
 - experiment_results.csv
 - queries_log.txt
 - errors_overhead.csv
